v0.9  
**Videoglancer.top - Convert YouTube video to PDF/PPTX**

**Instructions (Very Simple!)**

1. Run the 3 blocks below in order — just click ▶️ on each
2. Wait for the temporary Tunnel URLs to appear in Block 3
3. Click any tunnel URL to open the website


**⚠️ Important Notes:**
- Keep this notebook running (do not close the tab)
- The session will timeout after 90 minutes of inactivity
- Converted files are lost when Colab runtime resets. Only Google Drive saves persist.


# **Block 1: Download & Install Everything**

Run this cell to download the project and install all dependencies.

This will take 2-3 minutes.

In [ ]:
NOTEBOOK_VERSION = '0.9'
NOTEBOOK_UPDATE_MARKER = '/content/notebook_update_available.txt'


import os
import sys
import subprocess
import urllib.request

BASE_DIR = '/content/videoglancer-top'
ZIP_URL = 'https://raw.githubusercontent.com/2eerr/Videoglancer.top/main/videoglancer-top.zip'
ZIP_PATH = '/content/videoglancer-top.zip'
HAS_ERRORS = False
DOWNLOAD_FAILED = False

def run(cmd, desc, shell=False, timeout=120):
    print(f"  → {desc}...")
    import threading as _t
    try:
        p = subprocess.Popen(cmd, shell=shell, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        def _reader():
            for _l in p.stdout:
                print(f"     {_l.rstrip()}")
        _t.Thread(target=_reader, daemon=True).start()
        p.wait(timeout=timeout)
        if p.returncode == 0:
            print(f"  ✅ {desc}")
            return True
        print(f"  ⚠️  {desc} — failed (exit code {p.returncode})")
        return False
    except subprocess.TimeoutExpired:
        p.kill()
        print(f"  ⚠️  {desc} — timed out after {timeout}s")
        return False
    except FileNotFoundError:
        print(f"  ⚠️  {desc} — command not found on this system")
        return False
    except Exception as e:
        print(f"  ⚠️  {desc} — unexpected error: {e}")
        return False

# ── use uploaded zip if present ──
if os.path.exists(ZIP_PATH):
    print("📁 Found uploaded zip — using local copy")
else:
    # ── Download ──
    print("📥 Downloading videoglancer-top.zip...")
    try:
        req = urllib.request.Request(
            ZIP_URL,
            headers={
                'User-Agent': (
                    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                    '(KHTML, like Gecko) Chrome/123.0.0.0 Safari/537.36'
                ),
                'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
            },
        )
        with urllib.request.urlopen(req, timeout=30) as resp:
            with open(ZIP_PATH, 'wb') as f:
                f.write(resp.read())
        print('  ✅ Downloaded successfully')
    except Exception as e:
        print()
        print('=' * 70)
        print(f'❌  Failed to download from {ZIP_URL}')
        print('=' * 70)
        print(f'Error: {str(e)[:200]}')
        print()
        print('If the issue persists, the file may have moved.')
        print('=' * 70)
        DOWNLOAD_FAILED = True

if not DOWNLOAD_FAILED:
    # ── Extract ──
    print()
    print('📦 Extracting project...')
    if run(['unzip', '-q', '-o', ZIP_PATH, '-d', '/content/'], 'Extracting project'):
        os.chdir(BASE_DIR)
        if BASE_DIR not in sys.path:
            sys.path.insert(0, BASE_DIR)
        print('  ✅ Project extracted to /content/videoglancer-top/')
    else:
        print('  ❌ The downloaded zip appears corrupted. Try again.')
        DOWNLOAD_FAILED = True

if not DOWNLOAD_FAILED:
    # Check if this notebook is outdated by comparing against the extracted project version
    try:
        with open(os.path.join(BASE_DIR, 'static', 'notebook-version.txt'), 'r') as f:
            latest_notebook = f.read().strip()
        if latest_notebook and latest_notebook != NOTEBOOK_VERSION:
            with open(NOTEBOOK_UPDATE_MARKER, 'w') as f:
                f.write(latest_notebook)
        else:
            try: os.remove(NOTEBOOK_UPDATE_MARKER)
            except: pass
    except Exception:
        try: os.remove(NOTEBOOK_UPDATE_MARKER)
        except: pass

    # ── Python packages ──
    print()
    print('📦 Installing Python dependencies (this takes 1–2 min)...')
    if not run(['pip', 'install', '-q', '-U', '-r', 'requirements.txt'], 'Core Python packages'):
        HAS_ERRORS = True
    if not run(['pip', 'install', '-q', '-U', 'yt-dlp', 'yt-dlp-ejs'], 'yt-dlp + yt-dlp-ejs'):
        HAS_ERRORS = True

    # ── FFmpeg ──
    print()
    print('🎬 Installing FFmpeg (video processing)...')
    if not run(['apt-get', 'install', '-y', '-qq', 'ffmpeg'], 'FFmpeg'):
        HAS_ERRORS = True

    # ── Deno ──
    print()
    print('🔧 Installing Deno (YouTube challenge solver)...')
    deno_ok = run('curl -fsSL https://deno.land/x/install/install.sh | sh', 'Deno', shell=True, timeout=120)
    if deno_ok:
        deno_path = os.path.expanduser('~/.deno/bin')
        if deno_path not in os.environ['PATH']:
            os.environ['PATH'] = deno_path + os.pathsep + os.environ['PATH']
    else:
        HAS_ERRORS = True

    # ── Cloudflare Tunnel ──
    print()
    print('🌐 Installing Cloudflare Tunnel (public URL)...')
    if run(['wget', '-q', 'https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb'], 'Downloading cloudflared'):
        if not run(['dpkg', '-i', 'cloudflared-linux-amd64.deb'], 'Installing cloudflared'):
            HAS_ERRORS = True
    else:
        HAS_ERRORS = True

    # -- Bore (fallback tunnel #2) --
    print()
    print('Installing Bore (alternative tunnel)...')
    bore_ok = run(['wget', '-q',
        'https://github.com/ekzhang/bore/releases/download/v0.5.0/bore-v0.5.0-x86_64-unknown-linux-musl.tar.gz',
        '-O', '/tmp/bore.tar.gz'], 'Downloading Bore')
    if bore_ok:
        if run(['tar', 'xzf', '/tmp/bore.tar.gz', '-C', '/usr/local/bin/'], 'Extracting Bore'):
            run(['chmod', '+x', '/usr/local/bin/bore'], 'Setting permissions')
            if os.path.exists('/tmp/bore.tar.gz'):
                os.remove('/tmp/bore.tar.gz')
            print('  Bore installed to /usr/local/bin/bore')
        else:
            HAS_ERRORS = True
    else:
        HAS_ERRORS = True

    # -- Bore Digital (bore.digital, HTTPS) --
    print()
    print('Installing Bore Digital (HTTPS tunnel)...')
    if run(['wget', '-q',
        'https://github.com/jkuri/bore/releases/download/v0.5.0/bore_linux_amd64',
        '-O', '/usr/local/bin/bore-digital'], 'Downloading Bore Digital'):
        run(['chmod', '+x', '/usr/local/bin/bore-digital'], 'Setting permissions')
        print('  Bore Digital installed to /usr/local/bin/bore-digital')
    else:
        HAS_ERRORS = True

    # ── Verify installed libraries ──
    print()
    print('🔍 Verifying installed libraries...')
    print()
    def _run_verification(cmd_list, label):
        """Run a command and print its first line of output."""
        try:
            result = subprocess.run(cmd_list, capture_output=True, text=True, timeout=10)
            if result.returncode == 0:
                first_line = result.stdout.strip().split(chr(10))[0]
                print(f'  ✅ {label}: {first_line}')
                return True
            else:
                print(f'  ⚠️  {label}: failed (exit {result.returncode})')
                return False
        except Exception as e:
            print(f'  ⚠️  {label}: {e}')
            return False

    all_ok = True
    if not _run_verification(['ffmpeg', '-version'], 'FFmpeg'):
        all_ok = False
    if not _run_verification(['deno', '--version'], 'Deno'):
        all_ok = False
    if not _run_verification(['yt-dlp', '--version'], 'yt-dlp'):
        all_ok = False
    if not _run_verification(['cloudflared', '--version'], 'cloudflared'):
        all_ok = False
    if not _run_verification(['bore', '--help'], 'bore'):
        all_ok = False
    if not _run_verification(['which', 'bore-digital'], 'bore-digital'):
        all_ok = False
        all_ok = False
    if all_ok:
        print('  ✅ All libraries verified successfully')
    else:
        print('  ⚠️  Some libraries failed verification — check above')
        HAS_ERRORS = True

print()
if DOWNLOAD_FAILED:
    print('⚠️  Block 1 failed — could not download project. Check the URL or upload the zip manually.')
elif HAS_ERRORS:
    print('⚠️  Block 1 completed with warnings — proceed to Block 2.')
else:
    print('✅ Block 1 complete! All tools installed successfully. Move to Block 2.')


# **Block 2: Mount Google Drive**

Run this cell to mount Google Drive for persistent file storage.

This is optional but recommended to save your converted files permanently.

In [ ]:
from google.colab import drive
import os
import time

DRIVE_MOUNT_PATH = '/content/drive'
LOCAL_OUTPUT_DIR = '/content/videoglancer-top/output/'

print('☁️  Mounting Google Drive...')
print()

if os.path.isfile(os.path.join(DRIVE_MOUNT_PATH, 'MyDrive', '.shortcut-targets-by-id')):
    print('  ✅ Google Drive is already mounted.')
else:
    print('  A browser tab will open asking for permission.')
    print('  Click "Connect to Google Drive" and then click "Continue" to sign in with Google.')
    print('  If nothing opens, look for a popup blocker notification.\n')

    try:
        drive.mount(DRIVE_MOUNT_PATH)
        time.sleep(1)

        if os.path.isdir(os.path.join(DRIVE_MOUNT_PATH, 'MyDrive')):
            print()
            print('  ✅ Google Drive mounted successfully!')
        else:
            print()
            print('  ⚠️  Mount completed but MyDrive is not accessible.')
            print(f'  Files will be saved locally in {LOCAL_OUTPUT_DIR}')

    except Exception as e:
        err_msg = str(e).lower()
        print()

        if any(word in err_msg for word in ['cancel', 'denied', 'rejected', 'access_blocked', 'user cancelled']):
            print('  ⚠️  Google Drive mount cancelled.')
        elif any(word in err_msg for word in ['auth', 'credential', 'token', 'permission', 'forbidden']):
            print('  ⚠️  Google Drive authentication failed.')
        elif any(word in err_msg for word in ['timeout', 'connection', 'network', 'eof', 'reset']):
            print('  ⚠️  Google Drive mount timed out.')
        else:
            print('  ⚠️  Could not mount Google Drive.')
            if str(e).strip():
                print(f'  Details: {str(e)[:200]}')

        print('  No problem — files will be saved locally instead.')

print()
print('  📍 Local output: ' + LOCAL_OUTPUT_DIR)
os.makedirs(LOCAL_OUTPUT_DIR, exist_ok=True)
print('  ✅ Now proceed to Block 3.')

# **Block 3: Start the Dev Server**

Run this cell to start the dev server and get public tunnel URLs.

This cell will never complete — the ▶️ icon will keep spinning. That is completely normal!

**⚠️ Important Notes:**
- Keep this cell running — do not stop it
- Click any public tunnel URL to open the website
- If one public tunnel URL does not work, try another


In [ ]:
import os
import subprocess
import threading
import time
import sys
import re
import socket
import shutil
import fcntl

PROJECT_DIR = '/content/videoglancer-top'

# Find an available port (try 5000-5010)
def find_free_port(start=5000, end=5010):
    for port in range(start, end + 1):
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            if s.connect_ex(('127.0.0.1', port)) != 0:
                return port
    return None

print('=' * 70)
print('STARTING DEV SERVER')
print('=' * 70)
FLASK_PORT = find_free_port()
if FLASK_PORT is None:
    print('Could not find an available port')
    raise SystemExit(1)

print(f'  Using port: {FLASK_PORT}')

# Start Flask
flask_started = threading.Event()

def start_flask():
    print(f'[1/2] Starting Flask server on port {FLASK_PORT}...')
    print('Flask logs will appear below:')
    env = os.environ.copy()
    env['FLASK_DEBUG'] = '0'
    env['FLASK_USE_RELOADER'] = '0'
    env['PORT'] = str(FLASK_PORT)
    try:
        process = subprocess.Popen(
            ['python', 'app.py'],
            cwd=PROJECT_DIR, env=env,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )
        flask_started.set()
        for line in process.stdout:
            print(f'  [FLASK] {line.strip()}')
        process.wait()
    except FileNotFoundError:
        print('Python not found. Did Step 1 complete?')
    except Exception as e:
        print(f'Failed to start Flask: {e}')

flask_thread = threading.Thread(target=start_flask, daemon=True)
flask_thread.start()
flask_started.wait(timeout=10)

# Wait for Flask to bind the port
for _ in range(60):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        if s.connect_ex(('127.0.0.1', FLASK_PORT)) == 0:
            break
    time.sleep(0.5)

print(f'\n[2/2] Starting all tunnel services in parallel (port {FLASK_PORT})...')
print('Waiting up to 60 seconds for URLs...')
print()

# Shared results dict (thread-safe)
results = {}
results_lock = threading.Lock()


def read_url_loop(proc, pattern, timeout=60, url_callback=None):
    """Read tunnel stdout looking for a URL matching the regex pattern.
    Uses non-blocking reads via fcntl."""
    fd = proc.stdout.fileno()
    fl = fcntl.fcntl(fd, fcntl.F_GETFL)
    fcntl.fcntl(fd, fcntl.F_SETFL, fl | os.O_NONBLOCK)

    output_buf = ''
    start = time.time()
    result = None

    while time.time() - start < timeout:
        try:
            chunk = proc.stdout.read(4096)
        except (ValueError, OSError, BlockingIOError):
            chunk = ''

        if chunk:
            output_buf += chunk
            while '\n' in output_buf:
                line, output_buf = output_buf.split('\n', 1)
                stripped = line.strip()
                if stripped:
                    print(f'     {stripped}')
                match = re.search(pattern, line)
                if match:
                    if url_callback:
                        result = url_callback(match)
                    else:
                        result = match.group(0)

            # Also search remaining buffer (TUI output may not end with newline)
            if result is None and output_buf.strip():
                match = re.search(pattern, output_buf)
                if match:
                    if url_callback:
                        result = url_callback(match)
                    else:
                        result = match.group(0)

        if proc.poll() is not None:
            try:
                remaining = proc.stdout.read()
                if remaining:
                    output_buf += remaining

                    # Also search remaining data for URL
                    if result is None:
                        match = re.search(pattern, output_buf)
                        if match:
                            if url_callback:
                                result = url_callback(match)
                            else:
                                result = match.group(0)
            except:
                pass
            break

        if result is not None:
            break

        time.sleep(0.1)


    return result


def run_cloudflare():
    """Start Cloudflare Tunnel and capture URL."""
    if not shutil.which('cloudflared'):
        return

    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{FLASK_PORT}', '--loglevel', 'info'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, universal_newlines=True
    )
    url = read_url_loop(proc, r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', timeout=60)
    with results_lock:
        results['cloudflare'] = {'url': url, 'proc': proc, 'name': 'Cloudflare'}


def run_bore():
    """Start Bore tunnel and capture URL."""
    bore_bin = shutil.which('bore')
    if not bore_bin:
        return

    proc = subprocess.Popen(
        [bore_bin, 'local', str(FLASK_PORT), '--to', 'bore.pub'],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    # Bore doesn't print a URL - it prints 'connected to server at bore.pub:PORT'
    url = read_url_loop(proc, r'bore\.pub:(\d+)', timeout=30,
                        url_callback=lambda m: f'http://bore.pub:{m.group(1)}')
    with results_lock:
        results['bore'] = {'url': url, 'proc': proc, 'name': 'Bore'}


def run_bore_digital():
    """Start Bore Digital tunnel (bore.digital, HTTPS)."""
    if not shutil.which('bore-digital'):
        return

    proc = subprocess.Popen(
        ['bore-digital', '-s', 'bore.digital', '-p', '2200',
         '-ls', 'localhost', '-lp', str(FLASK_PORT)],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1
    )
    url = read_url_loop(proc, r'https://[a-zA-Z0-9-]+\.bore\.digital', timeout=30)
    with results_lock:
        results['bore_digital'] = {'url': url, 'proc': proc, 'name': 'Bore Digital'}
# Start all 3 tunnels in parallel
tunnel_threads = [
    threading.Thread(target=run_cloudflare, daemon=True),
    threading.Thread(target=run_bore, daemon=True),
    threading.Thread(target=run_bore_digital, daemon=True),
]

for t in tunnel_threads:
    t.start()

# Wait for all tunnels to finish (or timeout)
for t in tunnel_threads:
    t.join(timeout=60)

print()
print()
print('=' * 70)
print('🌐 PUBLIC TUNNEL URLS')
print('=' * 70)
print()

domain_labels = {
    'cloudflare': 'trycloudflare.com',
    'bore_digital': 'bore.digital',
    'bore': 'bore.pub',
}
found_urls = []
for key in ['cloudflare', 'bore_digital', 'bore']:
    res = results.get(key)
    label = domain_labels.get(key, key.capitalize())
    if res and res['url']:
        found_urls.append(res)
        print(f'  {label}')
        print(f'  {res["url"]}')
        print()
    else:
        print(f'  {label}')
        print(f'  Failed (no URL within 60s)')
        print()

if found_urls:
    print('=' * 70)
    print('Open the URLs above to use the website')
    print('If one URL does not work, try another')
    print('=' * 70)
    print()
    print()
else:
    print('=' * 70)
    print('All tunnel services failed.')
    print('=' * 70)
    print(f'The app is still running at: http://127.0.0.1:{FLASK_PORT}')
    print('But you need a tunnel to access it from outside Colab.')
    print()
    print('Try restarting the runtime and running all blocks again.')

# Keep the process running so tunnels stay up
# Kill tunnels that didn't get a URL (they wasted resources)
for key, res in results.items():
    if res and not res['url'] and res['proc']:
        try:
            res['proc'].terminate()
        except:
            pass

# Wait for any successful tunnel process to keep the notebook alive


for key in ['cloudflare', 'bore_digital', 'bore']:
    res = results.get(key)
    if res and res['url'] and res['proc']:
        res['proc'].wait()
        break
